In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
root = "/ptmp/rfechner/out/exp07_rollouts/"
sources = [os.path.join(root, f) for f in os.listdir(root)]
sources

['/ptmp/rfechner/out/exp07_rollouts/qwen2.5_7b__grpo',
 '/ptmp/rfechner/out/exp07_rollouts/qwen3_8b__grpo',
 '/ptmp/rfechner/out/exp07_rollouts/llama_3.1_8b_instruct__grpo']

In [3]:
dfs = []
for source in sources:
    with open(os.path.join(source, 'val_jsonl/0_rollouts.jsonl'), 'r') as file:
        df = pd.read_json(file, lines=True)
        df['source'] = [source.split('/')[-1].split('__')[0]] * len(df)
        dfs.append(df)

In [4]:
dfs[0]

,input,output,gts,score,step,reward,acc,source
0,system\nYou are a helpful assistant.\nuser\nCo...,"The point $(0,3)$ in rectangular coordinates i...","\left( 3, \frac{\pi}{2} \right)",1,0,1,1,qwen2.5_7b
1,system\nYou are a helpful assistant.\nuser\nCo...,"To convert the point $(0,3)$ from rectangular ...","\left( 3, \frac{\pi}{2} \right)",0,0,0,0,qwen2.5_7b
2,system\nYou are a helpful assistant.\nuser\nCo...,"To convert the point \((0,3)\) from rectangula...","\left( 3, \frac{\pi}{2} \right)",1,0,1,1,qwen2.5_7b
3,system\nYou are a helpful assistant.\nuser\nCo...,"To convert the point $(0,3)$ from rectangular ...","\left( 3, \frac{\pi}{2} \right)",0,0,0,0,qwen2.5_7b
4,system\nYou are a helpful assistant.\nuser\nCo...,"To convert the point $(0,3)$ from rectangular ...","\left( 3, \frac{\pi}{2} \right)",0,0,0,0,qwen2.5_7b
...,...,...,...,...,...,...,...,...
7995,system\nYou are a helpful assistant.\nuser\nAl...,"To find the measure of angle $\angle AHB$, we ...",106^\circ,1,0,1,1,qwen2.5_7b
7996,system\nYou are a helpful assistant.\nuser\nAl...,"To find the measure of angle $\angle AHB$, we ...",106^\circ,1,0,1,1,qwen2.5_7b
7997,system\nYou are a helpful assistant.\nuser\nAl...,"To find the measure of angle AHB, we can follo...",106^\circ,0,0,0,0,qwen2.5_7b
7998,system\nYou are a helpful assistant.\nuser\nAl...,"To find \(\angle AHB\), we can use the propert...",106^\circ,1,0,1,1,qwen2.5_7b


In [16]:
import math
def pass_at_k(n, c, k):
    if c == 0:
        return 0.0
    if n <= k:
        return 1.0  # if we sample all items, success is guaranteed if c>0
    return 1 - (math.comb(n - c, k) / math.comb(n, k))

In [23]:
new_dfs = []
for df in dfs:
    # compute pass@8 per group
    pass8 = (
        df.groupby('input', sort=True)
        .apply(lambda g: pass_at_k(len(g), g['reward'].sum(), 8), include_groups=False)
        .rename('pass@8')
    )
    new_df = df.merge(pass8, how='left', on='input')
    new_dfs.append(new_df)

In [28]:
mask = (new_dfs[0]['pass@8'].to_numpy() * new_dfs[1]['pass@8'].to_numpy() * new_dfs[2]['pass@8'].to_numpy()) > 0

In [ ]:
new_dfs[0]['mask'] = mask

In [32]:
df = new_dfs[0]
solved = df[df['mask']]['input'].unique()

In [38]:
unique = set(df[df['mask']]['input'].to_list())